# Pandas sur les données du Titanic

Ces travaux pratiques introduisent les différents mécanismes de `pandas` en utilisant le [jeu de données Titanic](https://www.openml.org/d/40945) (un classique).

En cas de perte d'accès à cette ressource, vous pouvez télécharger le fichier ici : [titanic.csv](https://drive.google.com/file/d/1YjBNcjzKkG9dMVyH77EfdFwsqG6YX5wD/view?usp=drive_link)

In [ ]:
!wget https://www.openml.org/data/get_csv/16826755/phpMYEkMl -O titanic.csv
import pandas as pd
import matplotlib.pyplot as plt

## Chargement des données

* Commencez par charger les données du fichier `titanic.csv` en utilisant la commande [`pandas.read_csv`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html) dans un nouveau `DataFrame`.
* Afficher quelques lignes du `DataFrame`
* Afficher les informations générales du `DataFrame` avec [`pandas.DataFrame.describe`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.describe.html)

In [ ]:
# df =

### Solution

In [ ]:
# Chargement des données
df = pd.read_csv("titanic.csv")

# Affichage du dataframe
df

In [ ]:
df.describe()

## Isolement des colonnes contenant un `"?"`

Certaines valeurs sont incomplètes et contiennent la valeur `"?"`. Il faut les remplacer.

Pour commencer identifier toutes les colonnes qui contiennent un `"?"`. Utiliser le moyen de votre choix. Vous pouvez par exemple utiliser la fonction [`pandas.Series.unique`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.unique.html) sur chaque colonne et isoler celle contenant un `"?"`.

In [ ]:
# Votre code ici

### Solution

In [ ]:
# En utilisant un filtrage
count_missing = (df == "?").sum()
cols_to_fill = set(count_missing[count_missing > 0].index)
print("Liste des colonnes contenant un ?")
print(cols_to_fill)

print("--------------")
# En utilisant unique()
cols_to_fill = set()
for c in df.columns:
  if df[c].dtype == "O" and "?" in df[c].unique():
    cols_to_fill.add(c)
print("Liste des colonnes contenant un ?")
print(cols_to_fill)

## Remplacement des données par filtrage

Remplacez les données des colonnes contenant `"?"` en remplaçant par les valeurs suivantes :
* **`0`** pour `age`, `fare` et `body`
* **`unknown`** pour les autres colonnes : `embarked`, `home.dest`, `cabin` et `boat`

Pour changer les valeurs vous pouvez utiliser `df.loc[filtre, col] = new_val`.

In [ ]:
# # Remplacement des colonnes numériques
# ???

# # Remplacement des colonnes nominales
# ???

# # Changement des types des colonnes
# new_types={"fare": "float32",
#            "age": "float32",
#            "body": "int64"}
# df = df.astype(new_types)

## Solution

In [ ]:
# Remplacement des colonnes numériques
for col in {"age", "fare", "body"}:
  df.loc[df[col] == "?", col] = 0

# Remplacement des colonnes nominales
for  col in {'boat', 'cabin', 'embarked', 'home.dest'}:
  df.loc[df[col] == "?", col]="unknown"

# Changement des types des colonnes
new_types={"fare": "float32",
           "age": "float32",
           "body": "int64"}
df = df.astype(new_types)

## Un calcul simple

Pandas ne devrait maintenant plus avoir de secret pour vous.

Calculer le pourcentage de femmes et d'hommes ayant survécu au naufrage en proportion de leur sexe. C'est à dire

$$
  \frac{\text{Homme}_{\text{survivant}}}{\text{Homme}_{\text{total}}} \, \text{et} \, \frac{\text{Femme}_{\text{survivant}}}{\text{Femme}_{\text{total}}}
$$

In [ ]:
# Votre code ici

### Solution

In [ ]:
n_surv = df[df.survived == 1].sex.value_counts()
n_pass = df.sex.value_counts()
p_surv = n_surv / n_pass

print(f"Parmi les {n_pass['male']} hommes qui ont embarqué, "
      f"{n_surv['male']} ont survécu. Soit {p_surv['male']:.2%} des hommes")
print(f"Parmi les {n_pass['female']} femmes qui ont embarqué, "
      f"{n_surv['female']} ont survécu. Soit {p_surv['female']:.2%} des femmes")

## Une visualisation simple

Créer un barplot qui affiche le prix du ticket moyen en fonction de la classe du passager et de s'il a survécu.

Vous êtes libre de la disposition en séparant en deux `Axes` ou un seul.

In [ ]:
# Votre code

### Solution

In [ ]:
width = 0.35

df_survived_by_class = df.groupby(["survived","pclass"]).fare.mean().reset_index()
# df_survived_by_class = df_survived_by_class.reset_index()

fig, ax = plt.subplots()
ax.bar(df_survived_by_class[df_survived_by_class.survived == 1].pclass - width / 2,
       df_survived_by_class[df_survived_by_class.survived == 1].fare,
       width)
ax.bar(df_survived_by_class[df_survived_by_class.survived == 0].pclass + width / 2,
       df_survived_by_class[df_survived_by_class.survived == 0].fare,
       width)

ax.legend(["Survivant", "Décédé"])
ax.set_xticks([1, 2, 3])
ax.set_xticklabels(["$1^{re}$", "$2^{de}$", "$3^e$"])
ax.set_xlabel("Classe")
ax.set_ylabel("Prix du ticket")

plt.show()

## Nouvelle colonne `first_name`

Ajouter une colonne `first_name` à votre DataFrame qui contient le prénom du passager :
* Utilisez la colonne `name`
* Extrayez le prénom de celle-ci en utilisant [`pandas.Series.str.split`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.str.split.html). Le prénom se trouve avant la virgule `,` dans `name`
* Ajoutez le résultat à la colonne `first_name`


In [ ]:
# Votre code ici

### Solution

In [ ]:
df[["first_name", "title"]] = pd.DataFrame([first_name for first_name in df.name.str.split(",")]).loc[:,:1]
df["title"] = [fullname[0] for fullname in df.title.str.split(".")]
df

## Mapping

En utilisant [`pandas.Series.apply`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.map.html), transformez tous les prénoms de la colonne `first_name` en majuscules en mappant la fonction [`str.upper`](https://docs.python.org/fr/3/library/stdtypes.html#str.upper) de la [librairie standard](https://docs.python.org/fr/3/library/stdtypes.html) sur toute la colonne.

In [ ]:
# Votre code ici

### Solution

In [ ]:
df["first_name"] = df["first_name"].apply(str.upper)
df["first_name"]

## Age moyen par classe


Dans cet exercice, vous allez devoir calculer l'age moyen par classe et l'ajouter dans une nouvelle colonne `age_per_class` du DataFrame. Ce n'est évidemment pas idéal puisque nous introduisons de la redondance dans les données, mais l'objectif est d'utiliser les méthodes déjà vues puis de pratiquer les méthodes [`pandas.DataFrame.groupby`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.groupby.html) et [`pandas.DataFrame.join`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.join.html).

1. Dans un premier temps, utilisez toutes les techniques déjà vues pour créer cette nouvelle colonne (filtres, …) :
 * Initialisez la nouvelle colonne avec une valeur par défaut
 * Faites une boucle sur les classes et calculez la moyenne par classe, puis ajoutez la par filtrage
2. Dans un second temps, vous devrez réessayer le même exercice mais sans faire de boucle et en utilisant seulement [`pandas.DataFrame.groupby`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.groupby.html) et [`pandas.DataFrame.join`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.join.html) :
 * Faites un `DataFrame` en calculant un groupby sur `pclass` avec moyennage
 * Calculer la jointure entre votre `DataFrame` initial et celui de votre groupby.

In [ ]:
# # Calcule de la moyenne par classe avec une boucle
# df["age_per_class"] = # Valeur par defaut
# for v in df.pclass.unique():
#   ???

# print("RÉSULTAT PAR BOUCLE")
# print(df.age_per_class)
# print()

In [ ]:
# # Suppression de la colonne ajoutée pour recommencer l'exercice avec group_by
# df = df.drop("age_per_class", axis=1)

In [ ]:
# # Utilisation groupby
#
# # Calcul du groupby
# gb_pclass=
# # Jointure
# df = df.join(???)

# print("RÉSULTAT PAR GROUP_BY")
# print(df.age_per_class)
# print()

### Solution

In [ ]:
# Calcule de la moyenne avec une boucle
mean_age = df.age.mean()

df["age_per_class"] = mean_age
for v in df.pclass.unique():
  df.loc[df.pclass == v, "age_per_class"] = df[df.pclass == v].age.mean()

print("RÉSULTAT PAR BOUCLE")
print(df.age_per_class)
print()

In [ ]:
# Suppression de la colonne ajoutée pour recommencer l'exercice avec group_by
df = df.drop("age_per_class", axis=1)

In [ ]:
# Utilisation group_by
gb_pclass = df.groupby("pclass").age.mean()
df = df.join(gb_pclass, on="pclass", how="left", rsuffix="_per_class")
print(df)